# Matching por gm — NMOS / PMOS (LUT gm/ID)

- NMOS: um CSV por L; PMOS: um único CSV com todos os L. O L é lido do cabeçalho de cada coluna (`L_NMOS=6e-08`), não do nome do ficheiro.
- Eixo X de todas as curvas = VGS (PMOS: VSG), varrimento de 0 a 1 V com passo de 10 mV, a VDS = 200 mV.
- PMOS normalizado para valores positivos (`ID`, `VTH`, `VDSAT`, `VDS` em módulo; `VGS - VTH` como overdrive positivo em inversão forte). `CBS` é renomeado para `CSB`.
- `match_gm(device, gm)` devolve, para cada L, o ponto de operação (VGS) onde `GM = gm`, com todas as restantes grandezas interpoladas.

In [1]:
from pathlib import Path
import re
import numpy as np
import pandas as pd

DATA_DIR = Path("CSVs/")
NMOS_GLOB = "NMOS_L_*.csv"
PMOS_GLOB = "PMOS_L_*.csv"

HEADER = re.compile(r"^(?P<meas>.+?) \(L_(?P<dev>NMOS|PMOS)=(?P<L>[-+\d.eE]+)\) (?P<ax>[XY])$")
RENAME = {"CBS": "CSB", "VGS - VTH": "VOV"}
PMOS_ABS = ["ID", "VTH", "VDSAT", "VDS"]

In [2]:
def _read_csv(path):
    raw = pd.read_csv(path)
    curves = {}
    for col in raw.columns:
        m = HEADER.match(col.strip())
        if not m:
            continue
        key = (m["dev"], float(m["L"]))
        meas = RENAME.get(m["meas"], m["meas"])
        curves.setdefault(key, {}).setdefault(meas, {})[m["ax"]] = raw[col].dropna().to_numpy()
    frames = []
    for (dev, L), meass in curves.items():
        series = {meas: pd.Series(xy["Y"], index=xy["X"]) for meas, xy in meass.items()}
        df = pd.concat(series, axis=1).sort_index()
        df.index.name = "VGS"
        if dev == "PMOS":
            df[PMOS_ABS] = df[PMOS_ABS].abs()
            df["VOV"] = -df["VOV"]
        df.insert(0, "L", L)
        df.insert(0, "device", dev)
        frames.append(df.reset_index())
    return frames


def load_luts(data_dir=DATA_DIR):
    data_dir = Path(data_dir)
    frames = []
    for pattern in (NMOS_GLOB, PMOS_GLOB):
        for f in sorted(data_dir.glob(pattern)):
            frames += _read_csv(f)
    if not frames:
        raise FileNotFoundError(f"Nenhum CSV encontrado em {data_dir.resolve()}")
    lut = pd.concat(frames, ignore_index=True)
    return lut.sort_values(["device", "L", "VGS"]).reset_index(drop=True)


LUT = load_luts()
LUT.groupby("device")["L"].unique().apply(lambda a: [f"{x*1e9:g}n" for x in a])

device
NMOS    [60n, 120n, 180n, 240n, 300n, 500n, 750n, 1000...
PMOS    [60n, 120n, 180n, 240n, 300n, 500n, 750n, 1000...
Name: L, dtype: object

In [3]:
PROPS = ["ID", "GM/ID", "GM/GDS", "GDS", "GMB", "VTH", "VDSAT", "VOV", "VDS",
         "CGG", "CGS", "CGD", "CDB", "CSB", "CDD", "CSS"]


def _interp(x, y, xq):
    if np.all(y > 0):
        return float(np.exp(np.interp(xq, x, np.log(y))))
    return float(np.interp(xq, x, y))


def _branch_point(curve, gm, branch):
    g = curve["GM"].to_numpy()
    k = int(np.argmax(g))
    idx = np.arange(0, k + 1) if branch == "rising" else np.arange(k, len(g))[::-1]
    gb = g[idx]
    if len(idx) < 2 or not (gb.min() <= gm <= gb.max()):
        return None
    xs, xq = np.log(gb), np.log(gm)
    row = {"VGS": float(np.interp(xq, xs, curve["VGS"].to_numpy()[idx]))}
    for p in PROPS:
        if p in curve:
            row[p] = _interp(xs, curve[p].to_numpy()[idx], xq)
    return row


def match_gm(device, gm, L=None, falling=False):
    sub = LUT[LUT["device"] == device.upper()]
    if L is not None:
        sub = sub[np.isclose(sub["L"], L, rtol=1e-6)]
    branches = ["rising", "falling"] if falling else ["rising"]
    rows = []
    for Lval, curve in sub.groupby("L"):
        curve = curve.reset_index(drop=True)
        for b in branches:
            pt = _branch_point(curve, gm, b)
            if pt is not None:
                rows.append({"device": device.upper(), "L": Lval, "L_nm": round(Lval * 1e9, 6),
                             "branch": b, "GM": gm, **pt})
    cols = ["device", "L", "L_nm", "branch", "GM", "VGS"] + PROPS
    return pd.DataFrame(rows, columns=cols)


def gm_range(device):
    g = LUT[LUT["device"] == device.upper()].groupby("L")["GM"].agg(["min", "max"])
    g.index = (g.index * 1e9).rename("L_nm")
    return g

In [4]:
gm_range("NMOS"), gm_range("PMOS")

(                 min       max
 L_nm                          
 60.0    1.161648e-08  0.005900
 120.0   2.221618e-09  0.003935
 180.0   1.474927e-09  0.002918
 240.0   1.184300e-09  0.002321
 300.0   1.075700e-09  0.001945
 500.0   8.638907e-10  0.001258
 750.0   7.188651e-10  0.000867
 1000.0  6.239756e-10  0.000658
 1500.0  5.988599e-10  0.000451
 2000.0  5.493868e-10  0.000342,
                  min       max
 L_nm                          
 60.0    4.836087e-09  0.002395
 120.0   5.786074e-10  0.001419
 180.0   3.809048e-10  0.001014
 240.0   3.007000e-10  0.000789
 300.0   2.681087e-10  0.000645
 500.0   1.977034e-10  0.000398
 750.0   1.482453e-10  0.000267
 1000.0  1.182922e-10  0.000200
 1500.0  8.673546e-11  0.000132
 2000.0  6.830676e-11  0.000099)

In [5]:
pd.options.display.float_format = "{:.4g}".format
pd.options.display.max_columns = None
pd.options.display.width = 250

res = match_gm("PMOS", 1e-4)
res.drop(columns=["device", "L"])

,L_nm,branch,GM,VGS,ID,GM/ID,GM/GDS,GDS,GMB,VTH,VDSAT,VOV,VDS,CGG,CGS,CGD,CDB,CSB,CDD,CSS
0,60,rising,0.0001,0.4335,4.936e-06,20.26,8.78,1.139e-05,1.562e-05,0.5637,0.04625,-0.1302,0.2,5.927e-15,-3.116e-15,-2.213e-15,-1.394e-18,6.418e-18,2.26e-15,3.104e-15
1,120,rising,0.0001,0.4364,4.339e-06,23.05,21.36,4.682e-06,1.736e-05,0.5121,0.05648,-0.07572,0.2,8.408e-15,-5.073e-15,-2.235e-15,-5.66e-18,-3.929e-17,2.273e-15,5.092e-15
2,180,rising,0.0001,0.439,4.378e-06,22.84,27.22,3.673e-06,1.663e-05,0.4901,0.05959,-0.05109,0.2,1.16e-14,-7.809e-15,-2.257e-15,-1.148e-17,-4.427e-17,2.296e-15,7.816e-15
3,240,rising,0.0001,0.4428,4.489e-06,22.27,29.13,3.433e-06,1.615e-05,0.4768,0.06304,-0.03394,0.2,1.521e-14,-1.102e-14,-2.275e-15,-1.875e-17,-5.438e-17,2.328e-15,1.101e-14
4,300,rising,0.0001,0.4467,4.61e-06,21.69,30.21,3.31e-06,1.583e-05,0.4635,0.06451,-0.0168,0.2,1.954e-14,-1.507e-14,-2.294e-15,-2.807e-17,1.05e-16,2.373e-15,1.488e-14
5,500,rising,0.0001,0.464,5.186e-06,19.28,28.8,3.473e-06,1.531e-05,0.4408,0.07313,0.02325,0.2,3.568e-14,-3.049e-14,-2.376e-15,-6.868e-17,8.353e-16,2.604e-15,2.949e-14
6,750,rising,0.0001,0.4896,6.174e-06,16.2,23.54,4.248e-06,1.506e-05,0.4288,0.08756,0.06082,0.2,5.834e-14,-5.259e-14,-2.556e-15,-1.426e-16,2.041e-15,3.118e-15,5.032e-14
7,1000,rising,0.0001,0.5187,7.481e-06,13.37,17.56,5.694e-06,1.496e-05,0.4227,0.1051,0.09605,0.2,8.303e-14,-7.695e-14,-2.885e-15,-2.526e-16,3.533e-15,4.044e-15,7.321e-14
8,1500,rising,0.0001,0.5922,1.155e-05,8.66,7.796,1.283e-05,1.465e-05,0.4137,0.1547,0.1785,0.2,1.383e-13,-1.311e-13,-4.737e-15,-6.848e-16,7.808e-15,9.231e-15,1.243e-13


In [6]:
res.loc[res["GM/GDS"].idxmax()]

device         PMOS
L             3e-07
L_nm            300
branch       rising
GM           0.0001
VGS          0.4467
ID         4.61e-06
GM/ID         21.69
GM/GDS        30.21
GDS        3.31e-06
GMB       1.583e-05
VTH          0.4635
VDSAT       0.06451
VOV         -0.0168
VDS             0.2
CGG       1.954e-14
CGS      -1.507e-14
CGD      -2.294e-15
CDB      -2.807e-17
CSB        1.05e-16
CDD       2.373e-15
CSS       1.488e-14
Name: 4, dtype: object

**Notas**

- Para um dado gm, cada L tem em geral uma solução (desde que `gm <= gm_max(L)`), logo gm sozinho não fixa L: obtém-se um candidato por L e escolhe-se por outro critério (ganho `GM/GDS`, `ID`, `GM/ID`, capacidades, ...).
- gm não é monótono em VGS (pico perto de 0.8–0.9 V). Por omissão só se usa o ramo ascendente (até ao pico); `falling=True` acrescenta o ramo descendente.
- `match_gm("NMOS", gm, L=60e-9)` restringe a um L; podes filtrar o resultado com `res.query(...)`.
- gm vem em S para a largura total do ficheiro (`WF=2u`, `NF=5`, `M=1`).